# Waypoint Stage

A `WaypointStage` is an intermediate target: agents head for a position and are considered to have reached it once they come within a configurable `distance`.
It is created with `simulation.add_waypoint_stage(position, distance)` and chained to the next stage — usually an exit — with a fixed transition.
This notebook shows a single waypoint followed by an exit.

See {doc}`Object model & lifecycle </concepts/object_model>` for the full stage/journey lifecycle.

In [1]:
import pathlib

import jupedsim as jps
import shapely

# 20 m x 10 m corridor.
geometry = shapely.Polygon([(0, 0), (20, 0), (20, 10), (0, 10)])

trajectory_file = pathlib.Path("waypoints.sqlite")
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
)

# One waypoint in the middle of the room; agents must come within 0.5 m.
wp1 = simulation.add_waypoint_stage((10, 5), 0.5)
exit_id = simulation.add_exit_stage([(19, 4), (20, 4), (20, 6), (19, 6)])

# Wire waypoint → exit with a fixed transition.
journey = jps.JourneyDescription([wp1, exit_id])
journey.set_transition_for_stage(wp1, jps.Transition.create_fixed_transition(exit_id))
journey_id = simulation.add_journey(journey)

# Place 15 agents on the left side.
n_agents = 15
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (3, 0.5), (3, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=wp1,
            position=position,
            radius=0.12,
        )
    )

while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    simulation.iterate()

print(f"Done in {simulation.iteration_count()} iterations. Trajectories: {trajectory_file}")

Done in 2020 iterations. Trajectories: waypoints.sqlite


## Watch it

In [2]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Change the arrival `distance` from `0.5` to `2.0` and re-run. Agents will be counted as having reached the waypoint from further away, so they spread out more before heading to the exit.

## See also

- {doc}`Exit Stage </notebooks/fundamentals/03_exits>` — the simplest destination: agents are removed on arrival.
- {doc}`Journeys and Transitions </notebooks/fundamentals/07_journeys_transitions>` — chain multiple stages.
- {doc}`Object model & lifecycle </concepts/object_model>`.
- [jupedsim-scenarios](https://scenarios.jupedsim.org) — parameter sweeps and Monte-Carlo runs.